In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import preprocessing as p
from sklearn.model_selection import train_test_split


df = pd.read_csv('/usr/share/SI/alzheimers_disease_data.csv')

y = df['Diagnosis']
X = df.drop(['Diagnosis','DoctorInCharge'], axis = 1)

data_ss = p.StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(data_ss, y, test_size= 0.2, random_state=42)


In [ ]:
# Model z regularyzacją L2
model_with_regularizer = Sequential([
        Input(shape = (33,)),
        Dense(16, activation = 'relu', kernel_regularizer=l2(0.001)),
        Dense(8, activation = 'relu'),
        Dense(1, activation = 'sigmoid')
])

# Model bez regularyzacji
model_without_regularizer = Sequential([
        Input(shape = (33,)),
        Dense(16, activation = 'relu'),
        Dense(8, activation = 'relu'),
        Dense(1, activation = 'sigmoid')
])

In [ ]:
# Kompilacja modeli
model_with_regularizer.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_without_regularizer.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Trenowanie modeli
history_with_regularizer = model_with_regularizer.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
history_without_regularizer = model_without_regularizer.fit(X_test, y_test, epochs=10, batch_size=32, validation_split=0.2, verbose=0)


In [ ]:
# Wykresy porównawcze
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_with_regularizer.history['accuracy'], label='Trening')
plt.plot(history_with_regularizer.history['val_accuracy'], label='Test')
plt.title('Model z regularyzacją')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_without_regularizer.history['accuracy'], label='Trening')
plt.plot(history_without_regularizer.history['val_accuracy'], label='Test')
plt.title('Model bez regularyzacji')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()
